# INGARSS 2026 — Multi-Date Evaluation
### Persistence · Climatology · ConvLSTM · FlowCast  ×  VIS + WV

Runs all four methods on all test-split dates, aggregates mean ± std,
and generates the results table and per-horizon PSNR figure for the paper.

**Before running:** ensure the project is on Drive, ConvLSTM and FlowCast
checkpoints exist, and the runtime is set to **T4 GPU**
(Runtime → Change runtime type → T4 GPU).

## Setup — mount Drive, cd, install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/ISRO/ISRO A.1'  # ← edit if your path differs
%cd "$PROJECT_DIR"
!ls

In [ ]:
!pip install -q -r requirements.txt
!pip install -q imagecodecs huggingface_hub lpips==0.1.4
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## Check prerequisites — checkpoints and manifests

Run this cell to confirm every checkpoint needed by `eval_all_dates.py` is present.
If ConvLSTM is missing, run the training cell below first.

In [ ]:
import os, json

checks = [
    ('vis/manifest.csv',                 'VIS SD-VAE manifest (Stage 1)'),
    ('vis/checkpoints/best.pt',          'VIS ConvLSTM checkpoint (src/train.py)'),
    ('vis/checkpoints_fc/best_flow.pt',  'VIS FlowCast checkpoint'),
    ('wv/manifest.csv',                  'WV SD-VAE manifest'),
    ('wv/checkpoints/best.pt',           'WV ConvLSTM checkpoint'),
    ('wv/checkpoints_fc/best_flow.pt',   'WV FlowCast checkpoint'),
]

all_ok = True
for path, label in checks:
    exists = os.path.exists(path)
    sz = f'{os.path.getsize(path)/1e6:.1f} MB' if exists else ''
    status = f'✓ {sz}' if exists else '✗ MISSING'
    print(f'  {status:<20} {label}')
    if not exists:
        all_ok = False

print()
print('All prerequisites met ✅' if all_ok else '⚠️  Some files missing — see notes above')

## (Optional) Train ConvLSTM if checkpoint is missing

Skip if `vis/checkpoints/best.pt` and `wv/checkpoints/best.pt` already exist.
~6 h on T4 per channel.

In [ ]:
# Uncomment to train — runs ~6 h on T4 per channel
# !python -m src.train --config vis/config.yaml
# !python -m src.train --config wv/config.yaml

## VIS — Run multi-date evaluation (all 4 methods)

In [ ]:
# Single-sample run: fast, no CRPS (use for quick iteration)
!python -m src.eval_all_dates \
    --config    vis/config.yaml \
    --config-fc vis/config_fc.yaml \
    --samples   1 \
    --skip-existing

In [ ]:
# 8-sample ensemble: slower (~30 extra min), adds CRPS for FlowCast
# !python -m src.eval_all_dates \
#     --config    vis/config.yaml \
#     --config-fc vis/config_fc.yaml \
#     --samples   8 \
#     --skip-existing

## WV — Run multi-date evaluation

In [ ]:
!python -m src.eval_all_dates \
    --config    wv/config.yaml \
    --config-fc wv/config_fc.yaml \
    --samples   1 \
    --skip-existing

## Results table — VIS

In [ ]:
import json, pandas as pd

with open('vis/outputs/eval_all_dates.json') as f:
    ev = json.load(f)

SHOW = ['psnr', 'ssim', 'CSI_M', 'HSS_M', 'FAR_M', 'crps_mean']
rows = []
for method in ev['methods']:
    agg = ev['aggregated'].get(method, {})
    row = {'method': method}
    for k in SHOW:
        if k in agg:
            row[k] = f"{agg[k]['mean']:.3f} ± {agg[k]['std']:.3f}"
        else:
            row[k] = '—'
    rows.append(row)

df = pd.DataFrame(rows).set_index('method')
print(f"VIS — test dates: {ev['test_dates']}")
df

## Results table — WV

In [ ]:
with open('wv/outputs/eval_all_dates.json') as f:
    ev_wv = json.load(f)

rows = []
for method in ev_wv['methods']:
    agg = ev_wv['aggregated'].get(method, {})
    row = {'method': method}
    for k in SHOW:
        if k in agg:
            row[k] = f"{agg[k]['mean']:.3f} ± {agg[k]['std']:.3f}"
        else:
            row[k] = '—'
    rows.append(row)

df_wv = pd.DataFrame(rows).set_index('method')
print(f"WV — test dates: {ev_wv['test_dates']}")
df_wv

## Per-horizon PSNR curves (VIS)

In [ ]:
import json, matplotlib.pyplot as plt, numpy as np

with open('vis/outputs/eval_all_dates.json') as f:
    ev = json.load(f)

HORIZON_LABELS = ['0-11\n(0-6 h)', '12-23\n(6-12 h)', '24-35\n(12-18 h)', '36-47\n(18-24 h)']
HORIZON_KEYS   = ['0-11', '12-23', '24-35', '36-47']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (channel_key, ev_data, title) in zip(
    axes,
    [('vis', ev, 'VIS — Per-horizon PSNR'),
     ('wv',  ev_wv, 'WV — Per-horizon PSNR')]):

    for method in ev_data['methods']:
        psnr_vals = []
        for hk in HORIZON_KEYS:
            vals = []
            for date_res in ev_data['per_date'][method]:
                bh = date_res.get('by_horizon', {})
                if hk in bh and 'psnr' in bh[hk]:
                    vals.append(bh[hk]['psnr'])
            psnr_vals.append(np.mean(vals) if vals else float('nan'))

        # only plot if we have at least some data
        if any(not np.isnan(v) for v in psnr_vals):
            marker = 'o' if 'flowcast' in method else ('s' if 'convlstm' in method else '^')
            linestyle = '-' if 'flowcast' in method else ('--' if 'convlstm' in method else ':')
            ax.plot(range(4), psnr_vals, marker=marker, linestyle=linestyle,
                    linewidth=2, label=method)

    ax.set_xticks(range(4))
    ax.set_xticklabels(HORIZON_LABELS)
    ax.set_xlabel('Forecast horizon (steps / hours)')
    ax.set_ylabel('PSNR (dB)')
    ax.set_title(title)
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('vis/outputs/per_horizon_psnr_all_methods.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: vis/outputs/per_horizon_psnr_all_methods.png')

## Combined LaTeX table (copy-paste into paper)

In [ ]:
import json, numpy as np

LATEX_KEYS = ['psnr', 'ssim', 'CSI_M', 'CRPS']
LATEX_HEADERS = ['PSNR $\\uparrow$', 'SSIM $\\uparrow$',
                  'CSI-M $\\uparrow$', 'CRPS $\\downarrow$']
METHOD_DISPLAY = {
    'persistence':    'Persistence',
    'climatology':    'Climatology',
    'convlstm':       'ConvLSTM (ours)',
    'flowcast_s1':    'FlowCast (1-sample)',
    'flowcast_s8':    'FlowCast (8-sample)',
}

def fmt(agg, key):
    k = 'crps_mean' if key == 'CRPS' else key
    if k not in agg:
        return '—'
    v = agg[k]
    return f"{v['mean']:.3f}$\\pm${v['std']:.3f}"

print('% ── VIS ────────────────────────────────────────────────')
with open('vis/outputs/eval_all_dates.json') as f:
    ev = json.load(f)
print('\\multicolumn{5}{l}{\\textit{VIS channel}} \\\\')
for m in ev['methods']:
    agg = ev['aggregated'].get(m, {})
    label = METHOD_DISPLAY.get(m, m)
    cells = ' & '.join(fmt(agg, k) for k in LATEX_KEYS)
    print(f'{label} & {cells} \\\\')

print()
print('% ── WV ─────────────────────────────────────────────────')
with open('wv/outputs/eval_all_dates.json') as f:
    ev_wv = json.load(f)
print('\\multicolumn{5}{l}{\\textit{WV channel}} \\\\')
for m in ev_wv['methods']:
    agg = ev_wv['aggregated'].get(m, {})
    label = METHOD_DISPLAY.get(m, m)
    cells = ' & '.join(fmt(agg, k) for k in LATEX_KEYS)
    print(f'{label} & {cells} \\\\')

## (Optional) Snapshot results to zip for download

In [ ]:
import os, shutil, glob, zipfile, datetime as dt

TS = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
SNAP = f'eval_snapshot_{TS}'
os.makedirs(SNAP, exist_ok=True)

for ch in ('vis', 'wv'):
    out = f'{ch}/outputs'
    if os.path.isdir(out):
        for fn in glob.glob(f'{out}/eval_*.json') + glob.glob(f'{out}/eval_*.csv'):
            shutil.copy2(fn, os.path.join(SNAP, f'{ch}_' + os.path.basename(fn)))
        fig = f'{out}/per_horizon_psnr_all_methods.png'
        if os.path.exists(fig):
            shutil.copy2(fig, os.path.join(SNAP, f'{ch}_per_horizon_psnr.png'))

zip_path = f'{SNAP}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(SNAP)):
        zf.write(os.path.join(SNAP, fn), arcname=fn)
sz = os.path.getsize(zip_path) / 1024
print(f'Snapshot zip: {zip_path}  ({sz:.1f} KB)')